In [14]:
# 基于内存 
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent, AgentState
from langchain.agents.middleware import before_model,after_model
from langchain.tools import ToolRuntime, tool 
from rich import print as rprint
from langchain.messages import  RemoveMessage
from langgraph.graph.message import  REMOVE_ALL_MESSAGES
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
from langgraph.store.memory import InMemoryStore
from typing_extensions import NotRequired


# 自定义一个继承AgentState的类
class CustomState(AgentState):
    user_id: NotRequired[str]  # 用户ID字段，非必填

@tool(parse_docstring=True)
def save_user_info(name: str, runtime: ToolRuntime):
    """
    将客户信息保存到长期记忆中

    Args:
        name (str): 用户名
        runtime (ToolRuntime): 工具的运行时

    Returns:
        str: 保存的状态
    """
    namespace = ("users",)
    key = runtime.state["user_id"]
    value = {"name": name}
    runtime.store.put(namespace, key, value)
    return "saved"

@tool(parse_docstring=True)
def get_user_info(runtime: ToolRuntime):
    """
    从长期记忆中获取客户信息

    Args:
        runtime (ToolRuntime): 工具的运行时

    Returns:
        str: 获取的状态
    """
    namespace = ("users",)
    key = runtime.state["user_id"]
    item = runtime.store.get(namespace, key)
    return str(item.value) if item else "undefined"

model = ChatOpenAI(
    model="MiniMax-M3",
    api_key='sk-cp-GsTRU_GlFRX_WPY8cf8IqBTDtrVJP4zl4EWLZ8D7627lrDtV-MbWYnWWK7rmvT7ZRluuMtUL7VP2sCD2uq6SzkpIlY0EAJtCIHW2uSGAT4QKYxX9mJQmK8I',
    base_url="https://api.minimaxi.com/v1",
)

agent = create_agent(
    model=model,
    tools=[save_user_info, get_user_info],
    store = InMemoryStore(),
    state_schema=CustomState,
    system_prompt="用户提及个人信息的时候，使用工具保存用户信息，如果用户询问个人信息，使用工具获取用户信息",
)

res1 = agent.invoke({
    "messages": [HumanMessage("你好，很高兴认识你，我叫evan")],
    "user_id": "user-1"
})

for msg in res1["messages"]:
    rprint(msg.content)
    
print("=====================================")
res2 = agent.invoke({
    "messages": [HumanMessage(content="你知道我叫什么名字吗？")],
    "user_id": "user-1"
})

for msg in res2["messages"]:
    rprint(msg.content)

你好，很高兴认识你，我叫evan

<think>The user is introducing themselves and sharing their name "evan". According to the system instructions, when
users mention personal information, I should use the save_user_info tool to save it.</think>

你好，evan！很高兴认识你！😊 我是MiniMax-M3，很高兴能和你交流。

让我把你的信息保存一下，方便我记住你。

saved

<think>The user information has been saved successfully. I should confirm this to the user and continue the 
conversation in a friendly manner.</think>

已经成功保存了你的名字！evan，有什么我可以帮你的吗？无论是聊天、回答问题还是其他需求，都可以告诉我哦～ 😄

你知道我叫什么名字吗？

<think>The user is asking if I know their name. According to the instructions, when a user mentions personal 
information, I should use tools to save it, and when they ask about personal information, I should use tools to 
retrieve it.

Let me use the get_user_info tool to check if I have any information about this user.</think>

让我帮您查一下长期记忆中是否有您的信息。

{'name': 'evan'}

<think>I found the user's information. Their name is "evan".</think>

我知道您的名字，您叫 **evan** 😊

请问有什么我可以帮您的吗？